<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/students.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

import sqlite3



# Create an in-memory SQLite database connection

conn = sqlite3.connect(':memory:')

cursor = conn.cursor()



print("Setup complete. Environment ready.")



# Define a messy, unnormalized dataset (0NF)

# Issues: Non-atomic values (multiple text books), redundant data, mixed concerns.

data_0nf = {

    "Student_ID": [101, 101, 102, 103, 104],

    "Student_Name": ["Alice", "Alice", "Bob", "Charlie", "David"],

    "Course_Code": ["CS101", "MATH201", "CS101", "CS102", "MATH201"],

    "Course_Name": ["Intro to CS", "Calculus I", "Intro to CS", "Data Structures", "Calculus I"],

    "Instructor_ID": ["INS_01", "INS_02", "INS_01", "INS_03", "INS_05"],

    "Instructor_Name": ["Dr. Smith", "Dr. Jones", "Dr. Smith", "Dr. Alan", "Dr. Jones"],

    "Instructor_Office": ["Room 401", "Room 502", "Room 401", "Room 401", "Room 502"],

    "Textbooks_Required": ["Python Basics, Intro to CLI", "Calculus Vol 1", "Python Basics, Intro to CLI", "Algorithms Vol 1", "Calculus Vol 1"],

    "Grade": ["A", "B", "A", "B+", "A-"]

}



df_0nf = pd.DataFrame(data_0nf)

df_0nf.to_sql('Unnormalized_Leasing', conn, index=False, if_exists='replace')



print("\n--- Unnormalized Data (0NF) ---")

df_0nf

Setup complete. Environment ready.

--- Unnormalized Data (0NF) ---


,Student_ID,Student_Name,Course_Code,Course_Name,Instructor_ID,Instructor_Name,Instructor_Office,Textbooks_Required,Grade
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,"Python Basics, Intro to CLI",A
1,101,Alice,MATH201,Calculus I,INS_02,Dr. Jones,Room 502,Calculus Vol 1,B
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,"Python Basics, Intro to CLI",A
3,103,Charlie,CS102,Data Structures,INS_03,Dr. Alan,Room 401,Algorithms Vol 1,B+
4,104,David,MATH201,Calculus I,INS_05,Dr. Jones,Room 502,Calculus Vol 1,A-


In [ ]:
# Flattening the Textbooks_Required column to ensure atomicity

df_1nf = df_0nf.assign(Textbooks_Required=df_0nf['Textbooks_Required'].str.split(', ')).explode('Textbooks_Required')



# Save to SQL

df_1nf.to_sql('Table_1NF', conn, index=False, if_exists='replace')



print("--- 1NF Data (Atomic values enforced) ---")

print(f"Row count increased from {len(df_0nf)} to {len(df_1nf)} due to flattening.")

df_1nf

--- 1NF Data (Atomic values enforced) ---
Row count increased from 5 to 7 due to flattening.


,Student_ID,Student_Name,Course_Code,Course_Name,Instructor_ID,Instructor_Name,Instructor_Office,Textbooks_Required,Grade
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Python Basics,A
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Intro to CLI,A
1,101,Alice,MATH201,Calculus I,INS_02,Dr. Jones,Room 502,Calculus Vol 1,B
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Python Basics,A
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Intro to CLI,A
3,103,Charlie,CS102,Data Structures,INS_03,Dr. Alan,Room 401,Algorithms Vol 1,B+
4,104,David,MATH201,Calculus I,INS_05,Dr. Jones,Room 502,Calculus Vol 1,A-


In [ ]:
# =========================
# STEP 2.5 : Convert to 2NF
# =========================

# Create Students table
df_students_2nf = df_1nf[['Student_ID', 'Student_Name']].drop_duplicates().reset_index(drop=True)

# Create Courses table
df_courses_2nf = df_1nf[
    ['Course_Code', 'Course_Name', 'Instructor_ID', 'Instructor_Name', 'Instructor_Office']
].drop_duplicates().reset_index(drop=True)

# Create Enrollment table
df_enrollments_2nf = df_1nf[
    ['Student_ID', 'Course_Code', 'Grade', 'Textbooks_Required']
].drop_duplicates().reset_index(drop=True)

# Save tables into SQL
df_students_2nf.to_sql('Students_2NF', conn, index=False, if_exists='replace')
df_courses_2nf.to_sql('Courses_2NF', conn, index=False, if_exists='replace')
df_enrollments_2nf.to_sql('Enrollments_2NF', conn, index=False, if_exists='replace')

print("\n--- 2NF Tables Created ---")

print("\n[Students_2NF]")
display(df_students_2nf)

print("\n[Courses_2NF]")
display(df_courses_2nf)

print("\n[Enrollments_2NF]")
display(df_enrollments_2nf)


--- 2NF Tables Created ---

[Students_2NF]


,Student_ID,Student_Name
0,101,Alice
1,102,Bob
2,103,Charlie
3,104,David



[Courses_2NF]


,Course_Code,Course_Name,Instructor_ID,Instructor_Name,Instructor_Office
0,CS101,Intro to CS,INS_01,Dr. Smith,Room 401
1,MATH201,Calculus I,INS_02,Dr. Jones,Room 502
2,CS102,Data Structures,INS_03,Dr. Alan,Room 401
3,MATH201,Calculus I,INS_05,Dr. Jones,Room 502



[Enrollments_2NF]


,Student_ID,Course_Code,Grade,Textbooks_Required
0,101,CS101,A,Python Basics
1,101,CS101,A,Intro to CLI
2,101,MATH201,B,Calculus Vol 1
3,102,CS101,A,Python Basics
4,102,CS101,A,Intro to CLI
5,103,CS102,B+,Algorithms Vol 1
6,104,MATH201,A-,Calculus Vol 1


In [ ]:
# Extract Instructors into a separate table to eliminate transitive dependency

df_instructors_3nf = df_courses_2nf[['Instructor_ID', 'Instructor_Name', 'Instructor_Office']].drop_duplicates().reset_index(drop=True)

# Clean up Courses table to only keep the Instructor_ID as a Foreign Key

df_courses_3nf = df_courses_2nf[['Course_Code', 'Course_Name', 'Instructor_ID']].drop_duplicates().reset_index(drop=True)



# Save to SQL

df_instructors_3nf.to_sql('Instructors_3NF', conn, index=False, if_exists='replace')

df_courses_3nf.to_sql('Courses_3NF', conn, index=False, if_exists='replace')



print("--- 3NF Schema Refinement ---")

print("\n[Courses_3NF] (No transitive dependency):")

display(df_courses_3nf)

print("\n[Instructors_3NF] (New isolated table):")

display(df_instructors_3nf)

--- 3NF Schema Refinement ---

[Courses_3NF] (No transitive dependency):


,Course_Code,Course_Name,Instructor_ID
0,CS101,Intro to CS,INS_01
1,MATH201,Calculus I,INS_02
2,CS102,Data Structures,INS_03
3,MATH201,Calculus I,INS_05



[Instructors_3NF] (New isolated table):


,Instructor_ID,Instructor_Name,Instructor_Office
0,INS_01,Dr. Smith,Room 401
1,INS_02,Dr. Jones,Room 502
2,INS_03,Dr. Alan,Room 401
3,INS_05,Dr. Jones,Room 502
